In [1]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost imbalanced-learn



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# %%
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()

    # Previous campaign - clean signals only
    df['prev_success']      = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted']   = (df['previous'] == 0).astype(int)
    df['pdays_clean']       = df['pdays'].apply(lambda x: 999 if x == -1 else x)
    df['prev_contacts_log'] = np.log1p(df['previous'])

    # Call duration - log only, let trees find cuts
    df['duration_log'] = np.log1p(df['duration'])

    # Balance - log + sign flag
    df['log_balance'] = np.log1p(df['balance'].clip(lower=0))
    df['is_debt']     = (df['balance'] < 0).astype(int)

    # Campaign - log only
    df['log_campaign'] = np.log1p(df['campaign'])

    # Month - cyclical (avoids arbitrary ordinal distance)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Only the most meaningful interactions
    df['long_call']           = (df['duration'] > 300).astype(int)
    df['long_call_x_success'] = df['long_call'] * df['prev_success']
    df['duration_x_prev']     = df['duration_log'] * df['prev_contacts_log']

    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA  shape:', TEST_DATA.shape)


TRAIN_DATA shape: (29839, 29)
TEST_DATA  shape: (19893, 29)


In [3]:
# %%
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']

num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value', unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index
    )
    num_df = df[num_cols].copy()
    return pd.concat([cat_enc, num_df], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    """Leak-free target encoding via cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc = X_te.copy()
    global_mean = y_tr.mean()

    for col in cols:
        oof     = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))

        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = (
                y_tr.iloc[fold_tr_idx]
                .groupby(X_tr[col].iloc[fold_tr_idx])
                .mean()
            )
            oof[fold_val_idx] = (
                X_tr[col].iloc[fold_val_idx]
                .map(means).fillna(global_mean).values
            )
            te_vals += (
                X_te[col].reset_index(drop=True)
                .map(means).fillna(global_mean).values
                / n_splits
            )

        X_tr_enc[col + '_te'] = oof
        X_te_enc[col  + '_te'] = te_vals

    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)

print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)
print('\nAll features:')
print(list(X_train_te.columns))


X_train_te shape: (29839, 37)
X_test_te  shape: (19893, 37)

All features:
['job', 'marital_status', 'education', 'default_loan', 'housing_loan', 'personal_loan', 'contact_type', 'poutcome', 'age', 'balance', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'prev_success', 'never_contacted', 'pdays_clean', 'prev_contacts_log', 'duration_log', 'log_balance', 'is_debt', 'log_campaign', 'month_sin', 'month_cos', 'long_call', 'long_call_x_success', 'duration_x_prev', 'job_te', 'marital_status_te', 'education_te', 'default_loan_te', 'housing_loan_te', 'personal_loan_te', 'contact_type_te', 'poutcome_te']


In [4]:
# %%
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

N_SPLITS  = 10
skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

X_arr    = X_train_te.values
X_te_arr = X_test_te.values

# Hyper-parameters - conservative depth + strong regularisation to reduce overfit
hgbm_params = {
    'learning_rate':     0.016782184919286597,
    'max_iter':          1117,
    'max_leaf_nodes':    26,
    'max_depth':         5,
    'min_samples_leaf':  72,
    'l2_regularization': 2.2767559805786015,
}

xgb_params = {
    'n_estimators':     899,
    'learning_rate':    0.044925311663262746,
    'max_depth':        4,
    'min_child_weight': 49,
    'subsample':        0.9066264844754309,
    'colsample_bytree': 0.9404726184518012,
    'reg_alpha':        0.5796743517191622,
    'reg_lambda':       2.9719182594187536,
    'gamma':            1.0300044484691284,
    'scale_pos_weight': scale_pos,
    'eval_metric':      'logloss',
    'random_state':     42,
    'n_jobs':           -1,
}

lgbm_params = {
    'n_estimators':      654,
    'learning_rate':     0.025695396965748477,
    'max_depth':         7,
    'num_leaves':        24,
    'min_child_samples': 23,
    'subsample':         0.8156760979769445,
    'colsample_bytree':  0.7770742352579232,
    'reg_alpha':         2.8693357936064383,
    'reg_lambda':        4.335019813134123,
    'class_weight':      'balanced',
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

cat_params = {
    'iterations':         1200,
    'learning_rate':      0.02,
    'depth':              6,
    'l2_leaf_reg':        3.0,
    'auto_class_weights': 'Balanced',
    'eval_metric':        'Logloss',
    'random_seed':        42,
    'verbose':            0,
}

# OOF + test containers
model_names = ['HGBM', 'XGB', 'LGBM', 'CAT']
oof_preds   = {name: np.zeros(len(y_train))   for name in model_names}
test_preds  = {name: np.zeros(len(X_test_te)) for name in model_names}

# Training loop
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m = HistGradientBoostingClassifier(
        class_weight='balanced', random_state=42,
        early_stopping=False, **hgbm_params
    )
    m.fit(X_tr, y_tr)
    oof_preds['HGBM'][val_idx]  = m.predict_proba(X_val)[:, 1]
    test_preds['HGBM']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m = XGBClassifier(**xgb_params)
    m.fit(X_tr, y_tr)
    oof_preds['XGB'][val_idx]   = m.predict_proba(X_val)[:, 1]
    test_preds['XGB']          += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m = LGBMClassifier(**lgbm_params)
    m.fit(X_tr, y_tr)
    oof_preds['LGBM'][val_idx]  = m.predict_proba(X_val)[:, 1]
    test_preds['LGBM']         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr)
    oof_preds['CAT'][val_idx]   = m.predict_proba(X_val)[:, 1]
    test_preds['CAT']          += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')

# Per-model OOF Balanced Accuracy via Youden J threshold
print('\nOOF Balanced Accuracy per model (Youden J threshold):')
for name in model_names:
    fpr, tpr, thresholds = roc_curve(y_train, oof_preds[name])
    j       = tpr - fpr
    best_t  = float(thresholds[np.argmax(j)])
    best_ba = balanced_accuracy_score(
        y_train, (oof_preds[name] >= best_t).astype(int)
    )
    print(f'  {name:5s}: BA={best_ba:.4f}  threshold={best_t:.4f}')


Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

OOF Balanced Accuracy per model (Youden J threshold):
  HGBM : BA=0.8700  threshold=0.4533
  XGB  : BA=0.8714  threshold=0.4284
  LGBM : BA=0.8719  threshold=0.3951
  CAT  : BA=0.8739  threshold=0.4034


In [5]:
# %%
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve
from lightgbm import LGBMClassifier
from scipy.stats import rankdata

# Step 1: Rank-normalise each model's OOF & test probs
# Removes calibration differences, makes stacking input uniform [0,1]
def rank_norm(arr):
    return rankdata(arr) / len(arr)

oof_rank  = np.column_stack([rank_norm(oof_preds[n])  for n in model_names])
test_rank = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

# Step 2: LGBM meta-model - trained on OOF only (no leakage)
meta_skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
oof_stack = np.zeros(len(y_train))

meta_params = dict(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

for fold, (tr_idx, val_idx) in enumerate(meta_skf.split(oof_rank, y_train)):
    meta = LGBMClassifier(**meta_params)
    meta.fit(oof_rank[tr_idx], y_train[tr_idx])
    oof_stack[val_idx] = meta.predict_proba(oof_rank[val_idx])[:, 1]

# Train final meta on all OOF for test prediction
final_meta = LGBMClassifier(**meta_params)
final_meta.fit(oof_rank, y_train)
test_stack = final_meta.predict_proba(test_rank)[:, 1]

# Step 3: BA-weighted average as second ensemble signal
oof_ba_scores = {}
for name in model_names:
    fpr, tpr, ths = roc_curve(y_train, oof_preds[name])
    j = tpr - fpr
    t = float(ths[np.argmax(j)])
    oof_ba_scores[name] = balanced_accuracy_score(
        y_train, (oof_preds[name] >= t).astype(int)
    )

weights = np.array([oof_ba_scores[n] for n in model_names])
weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-9)
weights /= weights.sum()

print('Model weights (BA-proportional):')
for n, w in zip(model_names, weights):
    print(f'  {n}: {w:.4f}')

oof_wavg  = oof_rank  @ weights
test_wavg = test_rank @ weights

# Step 4: Blend meta + weighted average
BLEND_W    = 0.6
oof_blend  = BLEND_W * oof_stack  + (1 - BLEND_W) * oof_wavg
test_blend = BLEND_W * test_stack + (1 - BLEND_W) * test_wavg

# Step 5: Youden's J threshold (exact maximiser of Balanced Accuracy)
fpr_b, tpr_b, thresholds_b = roc_curve(y_train, oof_blend)
j_b            = tpr_b - fpr_b
best_threshold = float(thresholds_b[np.argmax(j_b)])
best_ba        = balanced_accuracy_score(
    y_train, (oof_blend >= best_threshold).astype(int)
)

print(f'\nBlended OOF BA : {best_ba:.5f}')
print(f'Optimal threshold (Youden J): {best_threshold:.4f}')

# Step 6: Threshold neighbourhood
print('\nThreshold | #Pred-1 | OOF BA')
print('-' * 38)
for t in np.arange(
    max(0.01, best_threshold - 0.04),
    min(0.99, best_threshold + 0.05),
    0.005
):
    preds = (oof_blend >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    mark  = ' <- best' if abs(t - best_threshold) < 0.003 else ''
    print(f'  {t:.3f}   | {preds.sum():6d}  | {ba:.4f}{mark}')


Model weights (BA-proportional):
  HGBM: 0.0000
  XGB: 0.1913
  LGBM: 0.2675
  CAT: 0.5412

Blended OOF BA : 0.87352
Optimal threshold (Youden J): 0.6096

Threshold | #Pred-1 | OOF BA
--------------------------------------
  0.570   |   7675  | 0.8729
  0.575   |   7607  | 0.8723
  0.580   |   7536  | 0.8727
  0.585   |   7489  | 0.8722
  0.590   |   7433  | 0.8727
  0.595   |   7359  | 0.8731
  0.600   |   7287  | 0.8728
  0.605   |   7239  | 0.8729
  0.610   |   7199  | 0.8735 <- best
  0.615   |   7150  | 0.8725
  0.620   |   7098  | 0.8725
  0.625   |   7048  | 0.8728
  0.630   |   7005  | 0.8730
  0.635   |   6946  | 0.8722
  0.640   |   6896  | 0.8721
  0.645   |   6837  | 0.8719
  0.650   |   6784  | 0.8715
  0.655   |   6720  | 0.8714
  0.660   |   6659  | 0.8708


In [6]:
# %%
test_classes = (test_blend >= best_threshold).astype(int)

n1 = test_classes.sum()
n0 = (test_classes == 0).sum()
print(f'Prediction distribution - 0: {n0},  1: {n1}  '
      f'(pos-rate {n1/(n0+n1)*100:.1f}%)')
print(f'Blended OOF BA : {best_ba:.5f}')
print(f'Threshold used : {best_threshold:.4f}')


Prediction distribution - 0: 15127,  1: 4766  (pos-rate 24.0%)
Blended OOF BA : 0.87352
Threshold used : 0.6096


In [7]:
# %%
submission = pd.DataFrame({
    'id':           TEST_DATA.index,
    'subscription': test_classes
})
submission.to_csv('submission_lean.csv', index=False)
print('Saved!  Preview:')
print(submission.head())
print(submission['subscription'].value_counts())


Saved!  Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
subscription
0    15127
1     4766
Name: count, dtype: int64
